In [1]:
import pandas as pd
import numpy as np
import os
import shutil
import tensorflow as tf
from tensorflow.keras import layers, models

In [2]:
breeds = {
        1 : 'Abyssinian',
        2 : 'Bengal',
        3 : 'Birman',
        4 : 'Bombay',
        5 : 'British_Shorthair',
        6 : 'Egyptian_Mau',
        7 : 'Maine_Coon',
        8 : 'Persian',
        9 : 'Ragdoll',
        10 : 'Russian_Blue',
        11 : 'Siamese',
        12 : 'Sphynx'        
    }

train_df = pd.read_csv('dataset/annotations/trainval.txt', header=None, sep=' ')
train_df = train_df.iloc[(train_df.iloc[:,2] == 1).values,[0,3]]
train_df = train_df.rename(columns={0 : 'file_name', 3:'breed'})
train_df['breed_name'] = train_df['breed'].map(breeds)
train_df.drop_duplicates(keep=False,inplace=True)

train_set = train_df.sample(frac=0.8, random_state=42)
val_set = train_df.drop(train_set.index)


test_df = pd.read_csv('dataset/annotations/test.txt', header=None, sep=' ')
test_df = test_df.iloc[(test_df.iloc[:,2] == 1).values,[0,3]]
test_df = test_df.rename(columns={0 : 'file_name', 3:'breed'})
test_df['breed_name'] = test_df['breed'].map(breeds)
test_df.drop_duplicates(keep=False,inplace=True)


In [3]:
def move_images(source, destination, df):
    for index, row in df.iterrows():
        source_path = source+row['file_name']+'.jpg'
        destination_path = destination+row['breed_name']
        if not os.path.exists(destination_path):
            os.makedirs(destination_path)
        try:
            shutil.copy(source_path,destination_path)
        except Exception as e:
            print('error :',e)
        


move_images('dataset/images/','dataset/train/',train_set)
move_images('dataset/images/','dataset/validation/',val_set)
move_images('dataset/images/','dataset/test/',test_df)

In [7]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Load dataset
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "dataset/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "dataset/validation",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "dataset/test",
    image_size = IMG_SIZE,
    batch_size=BATCH_SIZE
)

num_classes = len(breeds)

normalization_layer = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

Found 950 files belonging to 12 classes.
Found 238 files belonging to 12 classes.
Found 1183 files belonging to 12 classes.


In [8]:
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 12)             │         1,548 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,170,508 (42.61 MB)

 Trainable params: 11,170,508 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

Epoch 1/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 18s 572ms/step - accuracy: 0.9147 - loss: 0.3519 - val_accuracy: 0.2479 - val_loss: 3.3456
Epoch 2/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 17s 571ms/step - accuracy: 0.9189 - loss: 0.2594 - val_accuracy: 0.3067 - val_loss: 3.6206
Epoch 3/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 18s 586ms/step - accuracy: 0.9432 - loss: 0.2105 - val_accuracy: 0.2269 - val_loss: 3.9325
Epoch 4/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 17s 572ms/step - accuracy: 0.9505 - loss: 0.1599 - val_accuracy: 0.2815 - val_loss: 4.0492
Epoch 5/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 17s 567ms/step - accuracy: 0.9600 - loss: 0.1274 - val_accuracy: 0.2647 - val_loss: 4.5809
Epoch 6/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 17s 563ms/step - accuracy: 0.9568 - loss: 0.1419 - val_accuracy: 0.2437 - val_loss: 4.8317
Epoch 7/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 17s 564ms/step - accuracy: 0.9642 - loss: 0.1234 - val_accuracy: 0.2857 - val_loss: 4.6339
Epoch 8/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 17s 566ms/step - accuracy: 0.9695 - loss: 0.1204 - val_accu

In [11]:
loss, acc = model.evaluate(test_ds)
print("Validation accuracy:", acc)

37/37 ━━━━━━━━━━━━━━━━━━━━ 5s 138ms/step - accuracy: 0.2020 - loss: 1062.7029
Validation accuracy: 0.20202873647212982


In [12]:
model.save("cat_model.keras")